<a href="https://colab.research.google.com/github/momar1-dot/ss-ai/blob/main/Exercise_1_and_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Task 1: Data Exploration: Design a pipeline to describe the audio from each species with similar (time-domain statistical) metrics as the one discussed in the Tutorial 2 - Use Case in AI: Vibration
Analysis:
– Root means square.
– Standard deviation.
– Crest factor.
– Average amplitude.
– Min/Max amplitude.

"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple
import zipfile
import os

import numpy as np
import soundfile as sf


# ----------------------------
# Data structures
# ----------------------------

@dataclass(frozen=True)
class AudioExample:
    """One audio sample with inferred class label."""
    path: Path
    label: str


# ----------------------------
# Dataset loading
# ----------------------------

class FolderDatasetLoader:
    """
    Loads audio files by scanning subdirectories.
    Each subdirectory name is treated as a class label.
    """

    def __init__(self, dataset_dir: Path) -> None:
        self.dataset_dir = dataset_dir

    def load(self) -> List[AudioExample]:
        """Scan dataset directory and load audio paths."""
        if not self.dataset_dir.exists():
            raise FileNotFoundError("Dataset directory not found")

        examples: List[AudioExample] = []

        for class_dir in sorted(self.dataset_dir.iterdir()):
            if not class_dir.is_dir():
                continue

            label = class_dir.name
            # Changed from *.wav to *.mp3
            for audio_file in class_dir.glob("*.mp3"):
                examples.append(AudioExample(path=audio_file, label=label))

        if not examples:
            raise RuntimeError("No audio files found in dataset")

        return examples


# ----------------------------
# Audio loading
# ----------------------------

class AudioLoader:
    """Loads audio into mono signals."""

    def __init__(self, max_seconds: float | None = None) -> None:
        self.max_seconds = max_seconds

    def load_mono(self, path: Path) -> np.ndarray:
        """Load audio file as mono float64 signal."""
        data, sr = sf.read(str(path), dtype="float64", always_2d=True)
        mono = data.mean(axis=1)

        if self.max_seconds is not None:
            mono = mono[: int(sr * self.max_seconds)]

        mono = mono - float(np.mean(mono))  # remove DC
        return mono


# ----------------------------
# Feature extraction
# ----------------------------

class TimeDomainFeatureExtractor:
    """Computes time-domain statistics."""

    @staticmethod
    def rms(x: np.ndarray) -> float:
        return float(np.sqrt(np.mean(x * x))) if x.size else 0.0

    @staticmethod
    def standard_deviation(x: np.ndarray) -> float:
        return float(np.std(x)) if x.size else 0.0

    @staticmethod
    def crest_factor(x: np.ndarray) -> float:
        if x.size == 0:
            return 0.0
        peak = float(np.max(np.abs(x)))
        rms_val = TimeDomainFeatureExtractor.rms(x)
        return peak / rms_val if rms_val != 0.0 else 0.0

    @staticmethod
    def average_amplitude(x: np.ndarray) -> float:
        return float(np.mean(np.abs(x))) if x.size else 0.0

    @staticmethod
    def min_amplitude(x: np.ndarray) -> float:
        return float(np.min(x)) if x.size else 0.0

    @staticmethod
    def max_amplitude(x: np.ndarray) -> float:
        return float(np.max(x)) if x.size else 0.0

    def extract(self, x: np.ndarray) -> Dict[str, float]:
        return {
            "rms": self.rms(x),
            "std": self.standard_deviation(x),
            "crest": self.crest_factor(x),
            "avg_amp": self.average_amplitude(x),
            "min_amp": self.min_amplitude(x),
            "max_amp": self.max_amplitude(x),
        }


# ----------------------------
# Aggregation
# ----------------------------

def aggregate_by_class(
    features: List[Dict[str, float]],
    labels: List[str],
) -> Dict[str, Dict[str, float]]:
    """Compute per-class average features."""
    grouped: Dict[str, List[Dict[str, float]]] = {}

    for feats, label in zip(features, labels):
        grouped.setdefault(label, []).append(feats)

    averages: Dict[str, Dict[str, float]] = {}
    for label, rows in grouped.items():
        averages[label] = {
            key: float(np.mean([r[key] for r in rows]))
            for key in rows[0].keys()
        }

    return averages


# ----------------------------
# Main routine
# ----------------------------

def main() -> None:
    dataset_zip_path = Path("/content/train-bird-audio.zip")
    dataset_dir = Path("train-bird-audio")

    # Check if the dataset directory exists, if not, try to unzip
    if not dataset_dir.exists() and dataset_zip_path.exists():
        print(f"Unzipping {dataset_zip_path}...")
        with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Unzipping complete.")
    elif not dataset_dir.exists() and not dataset_zip_path.exists():
        raise FileNotFoundError(f"Neither {dataset_dir} nor {dataset_zip_path} found. Please ensure the dataset is available.")


    loader = FolderDatasetLoader(dataset_dir)
    examples = loader.load()

    audio_loader = AudioLoader(max_seconds=10.0)
    extractor = TimeDomainFeatureExtractor()

    features: List[Dict[str, float]] = []
    labels: List[str] = []

    for ex in examples:
        signal = audio_loader.load_mono(ex.path)
        feats = extractor.extract(signal)
        features.append(feats)
        labels.append(ex.label)

    print("First 10 extracted feature rows:")
    for ex, feats in list(zip(examples, features))[:10]:
        print(f"{ex.label:8s} {ex.path.name:25s} {feats}")

    per_class_avg = aggregate_by_class(features, labels)

    print("\nPer-class average time-domain features:")
    for label in sorted(per_class_avg.keys()):
        avg = per_class_avg[label]
        print(
            f"{label:8s} | "
            f"RMS={avg['rms']:.6f} "
            f"STD={avg['std']:.6f} "
            f"Crest={avg['crest']:.6f} "
            f"AvgAmp={avg['avg_amp']:.6f} "
            f"Min={avg['min_amp']:.6f} "
            f"Max={avg['max_amp']:.6f}"
        )


import shutil
from pathlib import Path

# Define paths
source_zip_path = Path("/content/drive/MyDrive/train-bird-audio.zip")
destination_zip_path = Path("/content/train-bird-audio.zip")

# Check if the source file exists in MyDrive and copy it
if source_zip_path.exists():
    if not destination_zip_path.exists(): # Only copy if not already there
        print(f"Kopiëren van {source_zip_path} naar {destination_zip_path}...")
        shutil.copy(source_zip_path, destination_zip_path)
        print("Kopiëren voltooid.")
    else:
        print(f"{destination_zip_path} bestaat al, kopiëren overgeslagen.")
else:
    print(f"{source_zip_path} niet gevonden in Google Drive. Controleer het pad of upload het bestand.")

# Now call main() after ensuring the file is copied
if __name__ == "__main__":
    main()


Kopiëren van /content/drive/MyDrive/train-bird-audio.zip naar /content/train-bird-audio.zip...
Kopiëren voltooid.
Unzipping /content/train-bird-audio.zip...
Unzipping complete.
First 10 extracted feature rows:
aldfly   XC330449.mp3              {'rms': 0.004064126575859688, 'std': 0.004064126575859688, 'crest': 16.755473014104542, 'avg_amp': 0.0018905173282837801, 'min_amp': -0.06809636316772211, 'max_amp': 0.06683602817777884}
aldfly   XC241646.mp3              {'rms': 0.007070462714861878, 'std': 0.007070462714861878, 'crest': 7.641471649233241, 'avg_amp': 0.00399615979535839, 'min_amp': -0.054028740382577725, 'max_amp': 0.045074200959120445}
aldfly   XC317903.mp3              {'rms': 0.019717790657125596, 'std': 0.019717790657125596, 'crest': 14.354275823327738, 'avg_amp': 0.0070417039099350594, 'min_amp': -0.28152303602559814, 'max_amp': 0.2830346057190155}
aldfly   XC194088.mp3              {'rms': 0.01494310988364745, 'std': 0.01494310988364745, 'crest': 20.387345415388733, 'avg_

In [11]:
#!/usr/bin/env python3
"""
Task 3: Signal Processing (FIR filter bank denoising)

This script:
- Iterates over subdirectories (each = one bird species)
- Loads audio files directly from disk
- Applies an FIR filter bank to reduce environmental noise

Filter design (from scratch, no DSP libraries):
- Windowed-sinc FIR low-pass:
    h_lp[n] = 2fc * sinc(2fc(n-M)) * w[n]
  where fc is normalized cutoff (cutoff_hz / sample_rate),
  M = (num_taps - 1)/2, and w[n] is a Hamming window.
- Band-pass is built using:
    h_bp = h_lp(high) - h_lp(low)

Filter bank output is the sum of band-pass outputs:
    y[n] = Σ_i (x[n] * h_i[n])

"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple
import zipfile

import numpy as np
import soundfile as sf


# ----------------------------
# Data structures
# ----------------------------

@dataclass(frozen=True)
class AudioExample:
    """One audio sample with inferred class label."""
    path: Path
    label: str


@dataclass(frozen=True)
class FilterBand:
    """One passband [low_hz, high_hz] in Hz."""
    low_hz: float
    high_hz: float


# ----------------------------
# Dataset loading
# ----------------------------

class FolderDatasetLoader:
    """
    Loads audio files by scanning subdirectories.
    Each subdirectory name is treated as a class label.
    """

    def __init__(self, dataset_dir: Path, extension: str = "*.mp3") -> None:
        self.dataset_dir = dataset_dir
        self.extension = extension

    def load(self) -> List[AudioExample]:
        """Scan dataset directory and load audio paths."""
        if not self.dataset_dir.exists():
            raise FileNotFoundError("Dataset directory not found")

        examples: List[AudioExample] = []

        for class_dir in sorted(self.dataset_dir.iterdir()):
            if not class_dir.is_dir():
                continue

            label = class_dir.name
            for audio_file in class_dir.glob(self.extension):
                examples.append(AudioExample(path=audio_file, label=label))

        if not examples:
            raise RuntimeError("No audio files found in dataset")

        return examples


# ----------------------------
# Audio loading
# ----------------------------

class AudioLoader:
    """Loads audio into mono signals."""

    def __init__(self, max_seconds: float | None = None) -> None:
        self.max_seconds = max_seconds

    def load_mono(self, path: Path) -> Tuple[int, np.ndarray]:
        """Load audio file as mono float64 signal."""
        data, sr = sf.read(str(path), dtype="float64", always_2d=True)
        mono = data.mean(axis=1)

        if self.max_seconds is not None:
            mono = mono[: int(sr * self.max_seconds)]

        mono = mono - float(np.mean(mono))  # remove DC offset
        return sr, mono


# ----------------------------
# FIR filter bank (windowed-sinc)
# ----------------------------

class FIRFilterBank:
    """
    FIR filter bank for denoising.

    The filter bank consists of multiple band-pass filters.
    Each band-pass FIR is designed by:
        h_bp = h_lp(high) - h_lp(low)
    where h_lp is a windowed-sinc low-pass FIR.
    """

    def __init__(self, bands: List[FilterBand], num_taps: int) -> None:
        if num_taps % 2 == 0:
            raise ValueError("num_taps must be odd (linear-phase FIR).")
        self.bands = bands
        self.num_taps = num_taps

    @staticmethod
    def _hamming(n: int) -> np.ndarray:
        """Hamming window."""
        idx = np.arange(n, dtype=np.float64)
        return 0.54 - 0.46 * np.cos(2.0 * np.pi * idx / (n - 1))

    @staticmethod
    def _sinc(x: np.ndarray) -> np.ndarray:
        """Normalized sinc: sin(pi x)/(pi x), with sinc(0)=1."""
        y = np.ones_like(x, dtype=np.float64)
        nz = x != 0
        y[nz] = np.sin(np.pi * x[nz]) / (np.pi * x[nz])
        return y

    def _lowpass(self, cutoff_hz: float, sr: int) -> np.ndarray:
        """
        Windowed-sinc low-pass FIR.
        cutoff_hz must be in [0, sr/2].
        """
        if cutoff_hz <= 0.0:
            return np.zeros(self.num_taps, dtype=np.float64)

        nyquist = sr / 2.0
        cutoff = min(cutoff_hz, nyquist)
        fc = cutoff / sr  # normalized cutoff

        n = np.arange(self.num_taps, dtype=np.float64)
        m = (self.num_taps - 1) / 2.0

        h = 2.0 * fc * self._sinc(2.0 * fc * (n - m))
        h *= self._hamming(self.num_taps)

        # Normalize DC gain to 1
        s = float(np.sum(h))
        if s != 0.0:
            h /= s

        return h

    def _bandpass(self, low_hz: float, high_hz: float, sr: int) -> np.ndarray:
        """Band-pass FIR from two low-pass designs."""
        return self._lowpass(high_hz, sr) - self._lowpass(low_hz, sr)

    @staticmethod
    def _convolve_same(x: np.ndarray, h: np.ndarray) -> np.ndarray:
        """
        Convolve x with FIR h, returning output aligned with x length.
        Compensates for linear-phase group delay (len(h)-1)/2.
        """
        full = np.convolve(x, h, mode="full")
        delay = (len(h) - 1) // 2
        start = delay
        end = start + len(x)
        return full[start:end]

    def apply(self, x: np.ndarray, sr: int) -> np.ndarray:
        """
        Apply all band-pass filters and sum their outputs.
        """
        if x.size == 0:
            return x

        y = np.zeros_like(x, dtype=np.float64)

        for band in self.bands:
            h = self._bandpass(band.low_hz, band.high_hz, sr)
            y += self._convolve_same(x, h)

        # Optional normalization for safe audio range
        max_abs = float(np.max(np.abs(y))) if y.size else 0.0
        if max_abs > 1.0:
            y = y / max_abs

        return y


# ----------------------------
# Simple denoising diagnostics
# ----------------------------

def signal_energy(x: np.ndarray) -> float:
    """Return mean-square energy of the signal."""
    return float(np.mean(x * x)) if x.size else 0.0


def aggregate_by_class(values: List[float], labels: List[str]) -> Dict[str, float]:
    """Compute per-class average of scalar values."""
    grouped: Dict[str, List[float]] = {}
    for v, label in zip(values, labels):
        grouped.setdefault(label, []).append(v)

    return {label: float(np.mean(vs)) for label, vs in grouped.items()}


# ----------------------------
# Main routine
# ----------------------------

def main() -> None:
    dataset_zip_path = Path("/content/train-bird-audio.zip")
    dataset_dir = Path("train-bird-audio")

    if not dataset_dir.exists() and dataset_zip_path.exists():
        print(f"Unzipping {dataset_zip_path}...")
        with zipfile.ZipFile(dataset_zip_path, "r") as zip_ref:
            zip_ref.extractall(".")
        print("Unzipping complete.")
    elif not dataset_dir.exists():
        raise FileNotFoundError(
            f"Neither {dataset_dir} nor {dataset_zip_path} found."
        )

    loader = FolderDatasetLoader(dataset_dir, extension="*.mp3")
    examples = loader.load()

    audio_loader = AudioLoader(max_seconds=10.0)

    # Example filter bank bands (adjust based on your spectral analysis results)
    # Typical bird vocalizations often lie between ~1 kHz and ~10 kHz.
    bands = [
        FilterBand(800.0, 2500.0),
        FilterBand(2500.0, 6000.0),
        FilterBand(6000.0, 10000.0),
    ]
    filter_bank = FIRFilterBank(bands=bands, num_taps=401)

    before_energy: List[float] = []
    after_energy: List[float] = []
    labels: List[str] = []

    for ex in examples:
        sr, x = audio_loader.load_mono(ex.path)
        y = filter_bank.apply(x, sr)

        before_energy.append(signal_energy(x))
        after_energy.append(signal_energy(y))
        labels.append(ex.label)

    print("First 10 denoising checks (energy before/after):")
    for ex, e0, e1 in list(zip(examples, before_energy, after_energy))[:10]:
        print(
            f"{ex.label:8s} {ex.path.name:25s} "
            f"E_before={e0:.6e} E_after={e1:.6e}"
        )

    avg_before = aggregate_by_class(before_energy, labels)
    avg_after = aggregate_by_class(after_energy, labels)

    print("\nPer-class average energy before/after denoising:")
    for label in sorted(avg_before.keys()):
        print(
            f"{label:8s} | "
            f"E_before={avg_before[label]:.6e} "
            f"E_after={avg_after[label]:.6e}"
        )


if __name__ == "__main__":
    main()


First 10 denoising checks (energy before/after):
aldfly   XC330449.mp3              E_before=1.651712e-05 E_after=1.583346e-05
aldfly   XC241646.mp3              E_before=4.999144e-05 E_after=3.559973e-06
aldfly   XC317903.mp3              E_before=3.887913e-04 E_after=3.823821e-04
aldfly   XC194088.mp3              E_before=2.232965e-04 E_after=2.231676e-04
aldfly   XC157462.mp3              E_before=2.785043e-04 E_after=2.364552e-04
aldfly   XC195541.mp3              E_before=4.827371e-04 E_after=4.824841e-04
aldfly   XC189045.mp3              E_before=7.430237e-05 E_after=7.404123e-05
aldfly   XC189044.mp3              E_before=7.186243e-06 E_after=6.819985e-06
aldfly   XC317112.mp3              E_before=2.500402e-04 E_after=2.500511e-04
aldfly   XC189262.mp3              E_before=4.038668e-04 E_after=4.035825e-04

Per-class average energy before/after denoising:
aldfly   | E_before=1.058266e-03 E_after=6.247667e-04
amecro   | E_before=3.082974e-03 E_after=2.859221e-03
amered   | E_

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


After running the cell above and allowing Colab to mount your Google Drive, run the next cell to check if the file `train-bird-audio.zip` exists in your Google Drive at the specified path.

In [6]:
from pathlib import Path

file_path = Path('/content/drive/MyDrive/train-bird-audio.zip')

if file_path.exists():
    print(f"Het bestand '{file_path}' is gevonden in Google Drive.")
else:
    print(f"Het bestand '{file_path}' is NIET gevonden in Google Drive. Zorg ervoor dat het daar is geüpload.")

Het bestand '/content/drive/MyDrive/train-bird-audio.zip' is gevonden in Google Drive.
